# Diferencias en el consumo de potencia entre instrucciones (HARDWARE)

---
**Este no es el notebook completo, ver el fichero con (PRINCIPAL) en el nombre.**

---

Primero tenemos que seleccionar nuestro hardware, estableciendo las variables `SCOPETYPE` y `PLATFORM` al valor `'CWNANO'`.

In [ ]:
SCOPETYPE = 'CWNANO'
PLATFORM = 'CWNANO'

## Creación de un nuevo directorio

El siguiente bloque bash crea un nuevo directorio donde vamos a copiar el código del firmware `simpleserial-base` para luego modificarlo y compilarlo:

In [ ]:
%%bash
cd Firmwares
mkdir -p simpleserial-base-lab2 && cp -r simpleserial-base/* $_
cd simpleserial-base-lab2

## Compilando el Firmware

El siguiente bloque compila el firmware; vuelve a ejecutarlo cada vez que hagas cambios en el código:

In [ ]:
%%bash -s "$PLATFORM"
cd Firmwares/simpleserial-base-lab2
make PLATFORM=$1 CRYPTO_TARGET=NONE -j

## Conectando al Hardware

Vamos a configurar ChipWhisperer. Si todo va bien debe aparecer el mensaje: `INFO: Found ChipWhisperer😍`

In [ ]:
import chipwhisperer as cw
try:
    if not scope.connectStatus:
        scope.con()
except NameError:
    scope = cw.scope()

try:
    target = cw.target(scope)
except IOError:
    print("INFO: Caught exception on reconnecting to target - attempting to reconnect to scope first.")
    print("INFO: This is a work-around when USB has died without Python knowing. Ignore errors above this line.")
    scope = cw.scope()
    target = cw.target(scope)

print("INFO: Found ChipWhisperer😍")

if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
elif PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
    prog = cw.programmers.XMEGAProgrammer
else:
    prog = None
    
import time
time.sleep(0.05)
scope.default_setup()
def reset_target(scope):
    if PLATFORM == "CW303" or PLATFORM == "CWLITEXMEGA":
        scope.io.pdic = 'low'
        time.sleep(0.05)
        scope.io.pdic = 'high_z' #XMEGA doesn't like pdic driven high
        time.sleep(0.05)
    else:  
        scope.io.nrst = 'low'
        time.sleep(0.05)
        scope.io.nrst = 'high_z'
        time.sleep(0.05)

## Cargando el Firmware

El siguiente código graba el firmware en el objetivo (víctima). Tienes que ejecutarlo cada vez que compiles un nuevo firmware:

In [ ]:
cw.program_target(scope, prog, "Firmwares/simpleserial-base-lab2/simpleserial-base-{}.hex".format(PLATFORM))

## Capturando trazas de potencia

A continuación se muestra una función breve para capturar una traza de potencia. Estamos usando `cw.capture_trace()`, que es una función auxiliar para reemplazar lo que hicimos en el notebook anterior. Consulta la página de `readthedocs` para ver la documentación de la API: https://chipwhisperer.readthedocs.io/

El bloque de código DEFINE la función para capturar trazas; no captura ninguna por el momento.

In [ ]:
def capture_trace(_ignored=None):
    ktp = cw.ktp.Basic()
    key, text = ktp.next()
    return cw.capture_trace(scope, target, text).wave

---
**Agradecimientos / Referencias:**  
Este material está basado y adaptado a partir del notebook original [*Lab 2_1A - Instruction Power Differences (HARDWARE)*](https://github.com/newaetech/chipwhisperer-jupyter/blob/main/courses/sca101/Lab%202_1A%20-%20Instruction%20Power%20Differences%20(HARDWARE).ipynb) del repositorio oficial de [NewAE Technology](https://github.com/newaetech/chipwhisperer-jupyter).